In [1]:
%%capture
!pip install unsloth transformers peft accelerate bitsandbytes scikit-learn tqdm pandas

In [2]:
import os
import sys
import ast
import pandas as pd
import torch
import gc
from kaggle_secrets import UserSecretsClient

# Authenticate Hugging Face
try:
    os.environ["HF_TOKEN"] = "REDACTED_HF_TOKEN_SET_YOUR_OWN"
except Exception as e:
    print("Warning: HF_TOKEN secret not found in Kaggle Secrets.")

# Map out Kaggle paths based on your uploads
LOGLLM_CODE_DIR = "/kaggle/input/datasets/avyukthnunna/logllm-code"
sys.path.append(LOGLLM_CODE_DIR)

RAW_HDFS_LOG    = "/kaggle/input/datasets/avyukthnunna/hdfs-dataset/HDFS.log"
TEMPLATES_CSV   = "/kaggle/input/datasets/avyukthnunna/hdfs-dataset/HDFS_log_templates.csv"
LOGLLM_WEIGHTS  = "/kaggle/input/datasets/avyukthnunna/logllm-checkpoint/ft_model_HDFS"

# Working output files
PARSED_CSV      = "./parsed_sessions.csv"
SYNTHETIC_TEST_CSV = "./test.csv"
FINAL_EXPLANATIONS_CSV = "./final_explanations.csv"

In [3]:
import pandas as pd
import re
import csv
from datetime import datetime
from collections import defaultdict
import ast

print("1. Building Semantic Parser for LogLLM & Llama...")

templates = []
id_to_text = {} # Dictionary to store the English text for BERT

with open(TEMPLATES_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        event_id = row["EventId"].strip()
        template = row["EventTemplate"].strip()

        # Store the clean text with <*> for LogLLM
        clean_template = template.replace("[*]", "<*>")
        id_to_text[event_id] = clean_template

        # Build elastic regex
        parts = clean_template.split("<*>")
        escaped_parts = [re.escape(p) for p in parts]
        pattern_str = ".*?".join(escaped_parts)
        pattern_str = pattern_str.replace(r"\ ", r"\s+")

        compiled = re.compile(pattern_str, re.IGNORECASE | re.DOTALL)
        templates.append((event_id, compiled))

print(f"Loaded {len(templates)} templates with elastic regex.")

sessions = defaultdict(list)
block_pattern = re.compile(r"blk_-?\d+")
time_pattern = re.compile(r"^(\d{6})\s+(\d{6})")

matched_count = 0
with open(RAW_HDFS_LOG, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        if not line: continue

        block_match = block_pattern.search(line)
        if not block_match: continue
        block_id = block_match.group(0)

        event_id = None
        for eid, pat in templates:
            if pat.search(line):
                event_id = eid
                break

        if event_id:
            matched_count += 1
            ts_match = time_pattern.match(line)
            
            # Fetch the actual english text for BERT
            semantic_text = id_to_text[event_id] 
            
            if ts_match:
                dt = datetime.strptime(ts_match.group(1) + ts_match.group(2), "%y%m%d%H%M%S")
                # We save both the ID and the Text
                sessions[block_id].append((dt, event_id, semantic_text))
            else:
                sessions[block_id].append((None, event_id, semantic_text))

print(f"✅ Extracted {matched_count} events into {len(sessions)} unique block sequences.")
print("2. Formatting sequences...")

results = []
for block_id, events in sessions.items():
    events_sorted = sorted(events, key=lambda x: x[0] if x[0] is not None else datetime.max)
    
    # Llama 3.2 Explanation model expects IDs ("E1", "E2")
    id_sequence = [e[1] for e in events_sorted]
    
    # LogLLM (BERT) expects English text ("Receiving block...")
    text_sequence = [e[2] for e in events_sorted]

    timestamps = [e[0] for e in events_sorted if e[0] is not None]
    latency = (timestamps[-1] - timestamps[0]).total_seconds() if len(timestamps) >= 2 else 0.0

    results.append({
        "block_id": block_id,
        "sequence": str(id_sequence),        # Saved for Stage 2
        "text_sequence": str(text_sequence), # Saved for Stage 1
        "latency": latency
    })

df_parsed = pd.DataFrame(results)

def format_for_logllm(seq):
    try:
        events = ast.literal_eval(seq)
        if isinstance(events, list):
            return " ;-; ".join(events)
    except: pass
    return str(seq).replace(',', ' ;-; ')

if len(df_parsed) > 0:
    # FEED LOGLLM THE TEXT SEQUENCE
    df_parsed['Content'] = df_parsed['text_sequence'].apply(format_for_logllm)
    df_parsed['Label'] = 0 
    
    df_parsed[['Content', 'Label']].to_csv(SYNTHETIC_TEST_CSV, index=False)
    df_parsed.to_csv(PARSED_CSV, index=False)
    print(f"Success! Generated test.csv with {len(df_parsed)} sequences.")
else:
    print("❌ CRITICAL ERROR: Still 0 sequences matched.")

1. Building Semantic Parser for LogLLM & Llama...
Loaded 14 templates with elastic regex.
✅ Extracted 2000 events into 1994 unique block sequences.
2. Formatting sequences...
Success! Generated test.csv with 1994 sequences.


In [4]:

from torch.utils.data import DataLoader
from transformers import BertTokenizerFast
from customDataset import CustomDataset, CustomCollator
from model import LogLLM
import torch

print("Loading dataset into LogLLM pipeline...")
bert_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased', do_lower_case=True)
collator = CustomCollator(bert_tokenizer, max_seq_len=128, max_content_len=100)
# We feed it the file we just synthesized
dataset = CustomDataset(SYNTHETIC_TEST_CSV) 
dataloader = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=collator)

print("Initializing LogLLM Model...")
model = LogLLM(
    Bert_path='bert-base-uncased',
    Llama_path='meta-llama/Meta-Llama-3-8B', # Adjust to your exact base model if different
    ft_path=LOGLLM_WEIGHTS,
    is_train_mode=False
)
model.eval()

preds = []
print("Running Anomaly Classification...")
with torch.no_grad():
    # 🚨 THE MAGIC FIX: Force PyTorch to safely auto-cast BERT's float32 to Llama's bfloat16
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for batch in dataloader:
            inputs = {k: v.to(model.device) for k, v in batch['inputs'].items()}
            seq_positions = batch['seq_positions']
            
            output_tokens = model(inputs, seq_positions)
            
            for out in output_tokens:
                text = model.Llama_tokenizer.decode(out, skip_special_tokens=True).lower()
                if "anomalous" in text:
                    preds.append(1)
                else:
                    preds.append(0)

# Map predictions back to our master dataframe and filter out the normal ones
df_parsed['Prediction'] = preds
df_anomalies = df_parsed[df_parsed['Prediction'] == 1].copy()
print(f"Detection complete! LogLLM flagged {len(df_anomalies)} anomalous sequences.")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading dataset into LogLLM pipeline...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading peft model from /kaggle/input/datasets/avyukthnunna/logllm-checkpoint/ft_model_HDFS.
Running Anomaly Classification...
Detection complete! LogLLM flagged 1504 anomalous sequences.


In [5]:
print("Purging LogLLM from GPU memory...")
del model
del dataloader
del dataset
gc.collect()
torch.cuda.empty_cache()
print("VRAM cleared. Ready for the explanation model.")

Purging LogLLM from GPU memory...
VRAM cleared. Ready for the explanation model.


In [6]:
from unsloth import FastLanguageModel

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER_REPO = "kovidritesh/Llama-3.2-3B-FineTuned"

print("Loading explanation model...")
expl_model, expl_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    device_map="auto"
)

# Load your fine-tuned LoRA weights
expl_model.load_adapter(ADAPTER_REPO)
expl_model = FastLanguageModel.for_inference(expl_model)

results = []
print("Generating root-cause explanations...")

# Using .head(10) for the first test run so you don't wait hours. 
# Remove .head(10) to run on the entire anomaly dataset once you confirm it works.
for idx, row in df_anomalies.head(10).iterrows():
    prompt = f"Below is an anomalous HDFS log sequence. Explain what went wrong.\n\n### Sequence:\n{row['sequence']}\n\n### Latency:\n{row['latency']} sec\n\n### Explanation:\n"
    
    inputs = expl_tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = expl_model.generate(**inputs, max_new_tokens=300, use_cache=True)
    
    full_output = expl_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    explanation = full_output.split("### Explanation:\n")[-1].strip()
    
    results.append({
        "block_id": row['block_id'],
        "sequence": row['sequence'],
        "latency": row['latency'],
        "explanation": explanation
    })
    
    print(f"\n[Block ID]: {row['block_id']}")
    print(f"[Explanation]: {explanation}")
    print("-" * 50)

df_final = pd.DataFrame(results)
df_final.to_csv(FINAL_EXPLANATIONS_CSV, index=False)
print("Pipeline successfully finished. Explanations saved to disk.")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:165: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading explanation model...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...


[Block ID]: blk_7888946331804732825
[Explanation]: A block was allocated, but the packet responder terminated unexpectedly, causing the block map to be invalid. The NameSystem attempted to serve the block, but the block info was not found in the volume map, resulting in a block not found error. The sequence of events is: allocation, termination, invalid map, error trying to serve the block. The total latency is 0.0 seconds. An unexpected error occurred while serving the block. The failure is likely due to a network or disk issue. The exception is: BlockInfo not found in volume map for block: blk_-6743954263704342230. The exception is: BlockInfo not found in volume map for block: blk_-6743954263704342230. The exception is: BlockInfo not found in volume map for block: blk_-6743954263704342230. The exception is: BlockInfo not found in volume map for block: blk_-6743954263704342230. The exception is: BlockInfo not found in volume map for block: blk_-6743954263704342230. The exception is: 

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Block ID]: blk_2377150260128098806
[Explanation]: The packet responder terminated unexpectedly while serving a block. The entire trace completed in only 0.0 seconds. A hard pipeline failure occurred on the receiving DataNode. The failure is believed to be due to a corrupted block or a disk error on the DataNode. Block info: blk_7275117431111729349, NameSystem: NameSystem@0.0.0, PacketResponder: PacketResponder@0.0.0. The failure is indicated by the exception: java.lang.RuntimeException: Block info not found in volume map. The failure is also indicated by the unexpected termination of the packet responder. The entire trace completed in only 0.0 seconds. A hard failure occurred on the DataNode. The failure is believed to be due to a corrupted block or a disk error on the DataNode. The NameSystem terminated unexpectedly while serving a block. The failure is indicated by the exception: java.lang.RuntimeException: Block info not found in volume map. The failure is also indicated by the un

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Block ID]: blk_572492839287299681
[Explanation]: A BlockInfo is allocated for a block, but the block is immediately deleted. The delete operation is repeated three times. The block is then received by multiple sources, indicating a potential data replication issue. The received block is stored, but the block is later deleted again. The final state of the block is one of deletion, with the block being added to the invalid set. The repeated deletion operations and the addition to the invalid set suggest a failure mode where the block is repeatedly deleted and re-received, leading to inconsistencies in the block's metadata. The high latency of 0.0 seconds implies that the failure occurred almost immediately after the block was allocated. The repeated reception of the block and the repeated deletion operations indicate a potential issue with the block's replication or the network connection between the sources. The final state of the block is one of deletion, with the block being added t

In [7]:
import pandas as pd

# Load and display the final generation results
df_results = pd.read_csv("./final_explanations.csv")

# Set pandas options to show full text instead of truncating it
pd.set_option('display.max_colwidth', None)

# Display the dataframe
df_results

,block_id,sequence,latency,explanation
0,blk_38865049064139660,['E10'],0.0,"Block allocation at position 0. Block info: blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem, 1 block, BlockInfo [BLK, blk_-6676097437748571115, NameSystem,"
1,blk_-6952295868487656571,['E10'],0.0,"An unexpected error occurred at position 0.0. The packet responder terminated unexpectedly while serving a block. The failure is believed to be caused by a hard pipeline failure on the receiving DataNode. The failure is anomalous because the latency is 0.0 seconds, indicating the failure occurred almost immediately after the request was received. The failure is also anomalous because the failure signal was not followed by a recovery attempt. The exception is an empty string, indicating that the failure occurred without any error message being generated. The failure is believed to be caused by a hard pipeline failure on the receiving DataNode. The failure is anomalous because the failure occurred while serving a block, indicating that the block was not properly initialized or was corrupted. The failure is further anomalous because the failure was not followed by a recovery attempt, leaving the block in an inconsistent state. The failure is also anomalous because the exception is an empty string, indicating that the failure occurred without any error message being generated. The final state of the block is one of failure, with the block being deleted and the block info being updated. The block info is updated with the size of the block and the block being deleted. The block is deleted and the block info is updated. The block info is updated with the size of the block and the block being deleted. The block is deleted and the block info is updated. The block info is updated with the size of the block and the block being deleted. The block is deleted. The"
2,blk_7128370237687728475,['E6'],0.0,"The block allocation was successful, as evidenced by the block being added to the block map. However, the subsequent attempts to receive the block from different sources resulted in exceptions, indicating a failure in the data transfer process. The repeated attempts to receive the block, with the packet responder terminating, suggest a persistent issue with the block's transmission. The final state of the block is one of deletion, with the block being deleted multiple times, indicating a possible retry issue or a failure in the block's validation process. The high total latency of 0.0 seconds is unusual and suggests that the failure occurred almost immediately after the block allocation. The repeated deletion attempts imply a persistent issue with the block's storage or retrieval."
3,blk_8229193803249955061,['E10'],0.0,"The block was allocated by NameSystem, and the PacketResponder started receiving packets from multiple sources. The block was received by multiple threads, and the block map was updated multiple times. The block was received by multiple recipients, and the block was served to multiple clients. The block was deleted, but the deletion process failed. The block was added to the invalid set, and the block map was updated to reflect the deletion. The block was deleted again, and the block map was updated to reflect the final deletion.\n\n### Anomaly:\nThe block was deleted multiple times, and th

In [8]:
import pandas as pd

print("--- FIRST RAW LOG LINE ---")
with open('/kaggle/input/datasets/avyukthnunna/hdfs-dataset/HDFS.log', 'r') as f:
    print(f.readline().strip())

print("\n--- FIRST TEMPLATE LINE ---")
df_temp = pd.read_csv('/kaggle/input/datasets/avyukthnunna/hdfs-dataset/HDFS_log_templates.csv')
print(df_temp[['EventId', 'EventTemplate']].head(1).to_dict('records'))

--- FIRST RAW LOG LINE ---
081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating

--- FIRST TEMPLATE LINE ---
[{'EventId': 'E1', 'EventTemplate': '<*>:<*> Served block blk_<*> to /<*>'}]
